In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
%cd /content/drive/MyDrive/finetune_tinyllama_ai_ml_tutor

!pwd

!ls

/content/drive/MyDrive/finetune_tinyllama_ai_ml_tutor
/content/drive/MyDrive/finetune_tinyllama_ai_ml_tutor
data  notebooks  outputs  requirements.txt  src


In [2]:
!pip install -r requirements.txt

In [3]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [4]:
# DATASET_NAME = "databricks/databricks-dolly-15k"

path_files = {
        "train" : "data/raw/train.jsonl",
        "validate" : "data/raw/validate.jsonl",
        "test" : "data/raw/test.jsonl"
    }
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [5]:
from src.tokenizer.load_tokenizer import load_tokenizer

tokenizer = load_tokenizer(model_name=MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [6]:
from src.data.pipeline import pipeline

train, validate, test = pipeline(path_files)
print(train)
print(validate)
print(test)


Dataset({
    features: ['messages'],
    num_rows: 948
})
Dataset({
    features: ['messages'],
    num_rows: 118
})
Dataset({
    features: ['messages'],
    num_rows: 119
})


In [ ]:
dataset = dataset.select(range(500))
# print(dataset)

def get_token_length(example):
    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )

    return {
        "token_length": len(tokens["input_ids"])
    }

dataset = dataset.map(get_token_length)

max(dataset["token_length"].pop())
max(dataset["token_length"])


# for tl in dataset["token_length"]:
#     print(tl)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2268 > 2048). Running this sequence through the model will result in indexing errors


2268

In [8]:
from src.model.pipeline import pipeline

base_model, training_model = pipeline(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [9]:
from src.train.pipeline import pipeline

pipeline(dataset={"train":train, "validate":validate}, model=training_model, tokenizer=tokenizer)

average_tokens_across_devices is set to True but it is invalid when world size is1. Turn it to False automatically.


Tokenizing train dataset:   0%|          | 0/948 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2184 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/948 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/118 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/118 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.716400,0.691723
2,0.602900,0.667075
3,0.593200,0.664246


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


### TRAINING COOMPLETED ###


In [7]:
from src.inference.pipeline import pipeline

finetune_res, base_model_res = pipeline(base_model_name=MODEL_NAME, messages=train)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


question[0]['content']='How does PyTorch Autograd handle both forward pass and backward pass in a graph, and what are some key differences between these two components?'

finetuned_response="PyTorch Autograd is a system for automatic differentiation, where it maintains a computational graph to track the execution of an algorithm. This system is used to compute gradients of the loss function with respect to the model parameters, which are then used to update the model's weights during training. In this context, we'll focus on the difference between the forward and backward pass in PyTorch Autograd.\n\n**Definition**\n\nPyTorch Autograd is a system for automatic computation of gradients, which is fundamental to the training process. It keeps a computational graph to track the execution of an algorithm and provides a way to compute gradients of the loss function with respect to the model parameters."
basemodel_response='PyTorch Autograd handles both forward pass and backward pass in a gra